# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Marrwan1/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [3]:
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")
print("Token starts with:", token[:8])

con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{token}')")
print("✅ Secret created")

Token starts with: hf_JuDVO
✅ Secret created


In [4]:
MAR = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet('{MAR}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, row_count]
Index: []


## Fields: feature / label / context / excluded

**Features:** `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, `ga4_engaged_sessions` — these are page-performance signals used to score review priority.

**Label / proxy:** `future_decline_proxy` — a future outcome used as a proxy for refresh opportunity.

**Context:** `client_hash_id`, `content_hash_id`, `report_date`, `month` — used for grouping, identifying records, and time-based filtering or splitting, not as model features.

**Excluded:** `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, `ga4_data_available` — treated as data-availability metadata rather than core performance features. Future outcome-derived fields are also excluded to avoid leakage.

In [5]:
feature_frame = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions)                                AS gsc_impressions,
    SUM(gsc_clicks)                                     AS gsc_clicks,
    AVG(gsc_avg_position)                               AS gsc_avg_position,
    CASE WHEN SUM(gsc_impressions) = 0 THEN NULL
         ELSE SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
    END                                                 AS ctr,
    COUNT(DISTINCT report_date)                         AS days_with_data
FROM read_parquet('{MAR}')
WHERE gsc_data_available IS TRUE
  AND gsc_impressions > 0
GROUP BY client_hash_id, content_hash_id
LIMIT 10
""").df()

feature_frame

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,days_with_data
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,0.001073,31
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,0.000000,31
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,0.001066,31
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,0.002629,31
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,4.209227,0.002331,31
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,223.0,1.0,9.445635,0.004484,31
6,client_73cda7b4e4f265ea,content_1f380a642aed423b,96.0,1.0,6.014516,0.010417,31
7,client_73cda7b4e4f265ea,content_22c063002b7c1caf,314.0,1.0,9.155335,0.003185,31
8,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,7709.0,20.0,5.258331,0.002594,31
9,client_73cda7b4e4f265ea,content_20403327d8d9374c,3561.0,10.0,8.834415,0.002808,31


notes = """
FEATURE               | AVAILABLE WHEN?
gsc_impressions       | knowable at decision moment because it is aggregated
                      | from past daily GSC impressions in the feature month.

gsc_clicks            | knowable at decision moment because it is aggregated
                      | from past daily GSC clicks in the feature month.

gsc_avg_position      | knowable at decision moment because it is calculated
                      | from past GSC position observations in the feature month.

ctr                   | knowable at decision moment because it is calculated
                      | from past clicks and impressions in the feature month.

days_with_data        | knowable at decision moment because it counts the
                      | number of observed report dates in the feature month.
"""
print(notes)

## Verify it with queries

I will verify three facts about the March 2026 slice: the grain, the row count and date span, and GSC data availability.

In [6]:
grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet('{MAR}')
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print("Query 1 — duplicate grain rows:")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 — duplicate grain rows:
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, row_count]
Index: []


In [7]:
range_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet('{MAR}')
""").df()

print("Query 2 — row count and date span:")
print(range_check)

Query 2 — row count and date span:
   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


In [8]:
availability_check = con.sql(f"""
SELECT
    COUNT(*) AS available_rows
FROM read_parquet('{MAR}')
WHERE gsc_data_available IS TRUE
""").df()

print("Query 3 — GSC available rows:")
print(availability_check)

Query 3 — GSC available rows:
   available_rows
0         3611061


In [11]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

FEATURES = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ctr",
    "days_with_data"
]

X, y = df[FEATURES], df["label"]

# Honest model
clf = DecisionTreeClassifier(max_depth=4, random_state=42)
clf.fit(X, y)

honest_auc = roc_auc_score(
    y,
    clf.predict_proba(X)[:, 1]
)

print(f"Honest AUC: {honest_auc:.4f}")

# Leaky model
df["LEAKY_apr_clicks"] = df["total_clicks_apr"]

clf2 = DecisionTreeClassifier(max_depth=4, random_state=42)
clf2.fit(
    df[FEATURES + ["LEAKY_apr_clicks"]],
    y
)

leaky_auc = roc_auc_score(
    y,
    clf2.predict_proba(df[FEATURES + ["LEAKY_apr_clicks"]])[:, 1]
)

print(f"Leaky AUC: {leaky_auc:.4f}")

# Remove the leaked feature
df.drop(columns=["LEAKY_apr_clicks"], inplace=True)

print(f"\nLeaky column dropped. Honest AUC stays: {honest_auc:.4f}")

Honest AUC: 0.6962
Leaky AUC: 0.9673

Leaky column dropped. Honest AUC stays: 0.6962


### Leakage check

I intentionally added `total_clicks_apr`, which comes from the future outcome window, as a feature.

The honest model achieved an AUC of **0.6962**.

After adding the leaked future feature, AUC increased to **0.9673**, showing a large and unrealistic performance jump.

I then removed the leaked feature. The honest AUC remained **0.6962**.

This confirms that `total_clicks_apr` is not a valid feature at the decision moment and must be excluded from the honest feature set.

## Data limits / limitation

A limitation of this dataset slice is that GSC data is only available for part of the daily records. In March 2026, 3,611,061 out of 9,841,378 rows had `gsc_data_available IS TRUE`. Therefore, the feature frame only represents records with available GSC data and may not cover all content items or clients in the raw dataset.

Another limitation is that the label is a proxy based on future click growth rather than a direct measurement of whether a page truly needs a content refresh.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.